# Stage 2 Sentiment Training & Evaluation (MAMS)

**Dataset:** MAMS (8 categories)
**Embedding:** `p5-embed-v4` (Polonly + MAMS)

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os, sys, json, shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# --- Wire MAMS ---
# Assuming MAMS is available via p5-embed-v6-mams processed data or similar. 
# We will use the standalone MAMS processed data.
KAGGLE_INPUT = None
for candidate in ['/kaggle/input/p5-embed-v6-mams',
                  '/kaggle/input/datasets/duclm318/p5-embed-v6-mams',
                  '/kaggle/input/datasets/lcminhc/p5-embed-v6-mams']:
    if os.path.exists(candidate):
        KAGGLE_INPUT = candidate
        break
assert KAGGLE_INPUT, 'MAMS dataset not found'

os.makedirs('data/processed_mams', exist_ok=True)
for f_name in ['classification.jsonl', 'category_detection.jsonl', 'sentiment_records.jsonl']:
    src = f'{KAGGLE_INPUT}/processed_mams/{f_name}'
    if os.path.exists(src):
        shutil.copy(src, f'data/processed_mams/{f_name}')

# BUT since we want 8 categories, we must ensure sentiment_records.jsonl has 8 categories!
# Actually, the user noted MAMS should NOT use category mapping. 
# We need to run 01_prepare_data_mams.py to generate 8-cat sentiment_records.jsonl.
# We will download MAMS raw xml if needed, but let's assume raw MAMS is in a dataset:
RAW_MAMS = None
for candidate in ['/kaggle/input/mams-dataset', '/kaggle/input/datasets/duclm318/mams-dataset', '/kaggle/input/datasets/lcminhc/mams-dataset']:
    if os.path.exists(candidate):
        RAW_MAMS = candidate
        break
if RAW_MAMS:
    os.makedirs('MAMS', exist_ok=True)
    shutil.copytree(RAW_MAMS, 'MAMS', dirs_exist_ok=True)
    !python scripts/01_prepare_data_mams.py
else:
    print("Warning: Raw MAMS not found. Assuming data/processed_mams/ already has 8 categories!")

# --- Wire p5-embed-v4 ---
EMB = None
for candidate in ['/kaggle/input/p5-embed-v4',
                  '/kaggle/input/datasets/lcminhc/p5-embed-v4',
                  '/kaggle/input/datasets/duclm318/p5-embed-v4']:
    if os.path.exists(candidate):
        EMB = candidate
        break
assert EMB, 'Dataset p5-embed-v4 not found'
print(f'\nNB0 Input: {EMB}')

os.makedirs('checkpoints/embedding_mams', exist_ok=True)
shutil.copy(f'{EMB}/embedding_v4_s2_best.pt', 'checkpoints/embedding_mams/best.pt')
print(f'Embedding ckpt: {os.path.getsize("checkpoints/embedding_mams/best.pt") / 1e6:.1f} MB')

## 1. Build FAISS Index

In [ ]:
os.makedirs('indexes/mams', exist_ok=True)
!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding_mams/best.pt \
    --input data/processed_mams/sentiment_records.jsonl \
    --out_dir indexes/mams/

## 2. Train No-Retrieval (MAMS)

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
!python scripts/04b_train_stage2.py \
    --config configs/stage2_mams_noret.yaml \
    --no_retrieval

## 3. Train Retrieval + Aux Loss (MAMS)

In [ ]:
gc.collect()
torch.cuda.empty_cache()
!python scripts/04b_train_stage2.py \
    --config configs/stage2_mams_auxloss.yaml \
    --embedding_ckpt checkpoints/embedding_mams/best.pt \
    --index_dir indexes/mams/ \
    --retrieval_config configs/retrieval_v2.yaml

## 4. Evaluate Gold-Category Accuracy (MAMS)

In [ ]:
import subprocess
EVAL_EXPERIMENTS = [
    {"name": "No-Retrieval", "ckpt_dir": "checkpoints/stage2_mams_noret", "config": "configs/stage2_mams_noret.yaml", "no_retrieval": True},
    {"name": "Aux Loss", "ckpt_dir": "checkpoints/stage2_mams_auxloss", "config": "configs/stage2_mams_auxloss.yaml", "no_retrieval": False},
]

print("=" * 60)
print("GOLD-CATEGORY ACCURACY (MAMS 8 Categories)")
print("=" * 60)

for exp in EVAL_EXPERIMENTS:
    if not os.path.exists(f'{exp["ckpt_dir"]}/best.pt'):
        print(f'SKIP {exp["name"]} — checkpoint missing'); continue
    cmd = ["python", "scripts/06_evaluate_sentiment_only.py",
           "--stage2_ckpt", f'{exp["ckpt_dir"]}/best.pt',
           "--stage2_config", exp["config"],
           "--data_dir", "data/processed_mams"]
    if exp["no_retrieval"]:
        cmd.append("--no_retrieval")
    else:
        cmd += ["--embedding_ckpt", "checkpoints/embedding_mams/best.pt", "--index_dir", "indexes/mams/"]
    
    print(f'\n--- {exp["name"]} ---')
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)